In [1]:
import pandas as pd
import numpy as np
import random
import time

In [2]:
class ClassificationDatabase:
    """
    Stores product classification codes, their functions, and reference units.
    """

    def __init__(self, data):
        """
        Initialize the classification database.

        Args:
            data (list of dicts): List containing classification information.
        """
        self.data = pd.DataFrame(data)

    def get_sales_data(self, class_code):
        """
        Get sales data for products with a specific classification code.
        """
        print(f"  get_sales_data called for class code: '{class_code}'")  # debugging
        print(f"  all class codes in sales data: {self.data['class_code'].unique()}")  # debugging
        print(f"  Data types of class_code column: {self.data['class_code'].dtype}")  # debugging
        print(f"  Data type of class_code argument: {type(class_code)}")  # debugging
        sales_data = self.data[self.data["class_code"] == class_code]
        print(f"  Sales Data retrieved:\n{sales_data}")  # debugging
        return sales_data

    def get_class_info(self, class_code):
        """
        Retrieve the function and reference unit for a given class code.

        Args:
            class_code (str): The classification code (e.g., "HS 2517").

        Returns:
            tuple: (class_name, function, unit) if found, otherwise (None, None, None)
        """
        class_info = self.data.loc[self.data["class_code"] == class_code]
        if not class_info.empty:
            return (
                class_info.iloc[0]["class_name"],
                class_info.iloc[0]["function"],
                class_info.iloc[0]["unit"]
            )
        return None, None, None




In [3]:
class FootprintDatabase:
    """
    Stores reported footprints iteratively per year.
    """

    def __init__(self, year):
        self.year = year  # The year the footprint data is recorded
        self.data = pd.DataFrame()  # Create an empty DataFrame
        self.data["id"] = pd.Series(dtype="int")  # Add 'id' column
        self.data["impact_ids"] = pd.Series(dtype="object")  # Add 'impact_ids' column
        self.data["scores"] = pd.Series(dtype="float")  # Add 'scores' column
        #print("FootprintDatabase initialized. Data:") #debugging
        #print(self.data) #debugging

    def report(self, product_id, footprint):
        """
        Reports the latest footprint for a product.
        """
        #print("FootprintDatabase.report called. Current Data Columns:")
        #print(self.data.columns)  # Print the columns
        # Ensure 'id' column is integer
        if "id" in self.data:
            self.data["id"] = self.data["id"].astype(int)

        # Check if the product already has an entry
        if product_id in self.data["id"].values:
            # Update existing entry
            self.data.loc[self.data["id"] == product_id, "scores"] = footprint
        else:
            # Add a new entry
            new_entry = pd.DataFrame(
                {
                    "id": [product_id],
                    "impact_ids": ["CO2"],
                    "scores": [footprint],
                }
            )
            self.data = pd.concat([self.data, new_entry], ignore_index=True)

    def get_footprint(self, product_id):
        """
        Retrieve the footprint for a given product ID.
        """
        # Ensure 'id' column is integer
        if "id" in self.data:
            self.data["id"] = self.data["id"].astype(int)

        record = self.data.loc[self.data["id"] == product_id]
        return record["scores"].values[0] if not record.empty else 0




In [4]:
class Product:
    """
    Represents a product and its attributes.
    """

    def __init__(self, product_id, name, unit, company, class_code, primary_secondary, function_output, classification_db):
        self.product_id = product_id
        self.name = name
        self.unit = unit
        self.company = company  # Parent company
        self.class_code = class_code  # Classification code for system expansion
        self.primary_secondary = primary_secondary  # Whether it is a primary or secondary product
        self.function_output = function_output  # Quantity of function provided

        # Store a reference to the classification database instead of copying values
        self.classification_db = classification_db

    def get_classification_info(self):
        """
        Fetch function and reference unit from the classification database when needed.
        """
        class_info = self.classification_db.data.loc[self.classification_db.data["class_code"] == product.class_code]
        if not class_info.empty:
            function = class_info.iloc[0]["function"]
            reference_unit = class_info.iloc[0]["unit"]
        else:
            function = "Unknown"
            reference_unit = "Unknown"

        return {"function": function, "reference_unit": reference_unit}





In [101]:
class Company:
    """
    Represents a company with multi-level structure for environmental footprint reporting.
    """
    def __init__(self, name, year, purchases=None, sales=None, direct_impacts=None):
        self.name = name  # Name of the company
        self.year = year  # Reporting year
        self.purchases = pd.DataFrame(purchases) if purchases is not None else pd.DataFrame()
        self.sales = pd.DataFrame(sales) if sales is not None else pd.DataFrame()
        self.direct_impacts = pd.DataFrame(direct_impacts) if direct_impacts is not None else pd.DataFrame()
        self.sub_companies = []  # List of child Company objects
        self.products = []  # List of Product objects associated with this company
        self.latest_update = None
        self.num_updates = 0

    def add_product(self, product):
        """Add a product associated with this company."""
        self.products.append(product)

    def update_footprint(self, footprint_db_2024, footprint_db_2023, last_year_sales_db):
        """
        Update the company's footprint based on carbon balance.
        """
        print(f"Calculating footprint for {self.name}")

        # 1) Work on a fresh copy and reset index into a column
        df = footprint_db_2024.data.reset_index()
        # 2) Drop any duplicate columns that came from previous resets
        df = df.loc[:, ~df.columns.duplicated()]
        # 3) Ensure 'id' exists and is int
        df["id"] = df["id"].astype(int)

        product_ids = list(self.purchases.index.astype(int))
        print(f"  Expected product IDs: {product_ids}")

        existing_ids = set(df["id"])
        missing = [pid for pid in product_ids if pid not in existing_ids]
        if missing:
            print(f"  ⚠️ Warning: skipping missing IDs: {missing}")
        valid_ids = [pid for pid in product_ids if pid in existing_ids]

        print(f"  Footprint DB IDs available: {sorted(existing_ids)}")

        df_indexed = df.set_index("id", drop=False)
        total_impacts = (df_indexed.loc[valid_ids, "scores"] @
                         self.purchases.loc[valid_ids, :]) \
                        + self.direct_impacts.sum().sum()
        print(f"  Initial total impacts: {total_impacts}")

        primary_product_impacts = {}
        for product in self.products:
            print(f"\n  Processing product: {product.name} ({product.primary_secondary})")
            sales_vol = (self.sales.loc[product.product_id, "Sales"]
                         if product.product_id in self.sales.index else 0)

            if product.primary_secondary == "primary":
                primary_product_impacts[product.product_id] = total_impacts

            else:  # secondary
                avg_sub = self.get_average_footprint(
                    product.class_code, last_year_sales_db, footprint_db_2023
                )
                sub_imp = avg_sub * sales_vol * product.function_output
                for prim in self.products:
                    if (prim.primary_secondary == "primary" and
                        prim.product_id != product.product_id):
                        primary_product_impacts[prim.product_id] -= sub_imp
                product.footprint = sub_imp / sales_vol if sales_vol else 0
                print(f"    Assigned footprint (secondary): {product.footprint}")

        # Recalculate primaries
        for product in self.products:
            if product.primary_secondary == "primary":
                vol = (self.sales.loc[product.product_id, "Sales"]
                       if product.product_id in self.sales.index else 0)
                product.footprint = (primary_product_impacts[product.product_id] / vol
                                     if vol else 0)
                print(f"    Recalculated footprint (primary): {product.footprint}")

        # Company‐level weighted average
        num = sum(p.footprint * p.function_output for p in self.products)
        den = sum(p.function_output for p in self.products) or 1
        self.latest_update = float(num / den)

        self.report_footprint(footprint_db_2024)
        print(f"  Final footprint for {self.name}: {self.latest_update}")

    def check_update_needed(self, footprint_db_2024, last_year_sales_db):
        """
        Check if a footprint update is needed for any products in the company.
        """
        print(f"\nChecking updates for {self.name}")

        # Copy and reset index safely
        df = footprint_db_2024.data.reset_index()
        df = df.loc[:, ~df.columns.duplicated()]
        df["id"] = df["id"].astype(int)
        df_indexed = df.set_index("id", drop=False)

        purchase_ids = list(self.purchases.index.astype(int))
        print(f"  self.purchases.index: {purchase_ids}")
        print(f"  Columns available: {df.columns.tolist()}")

        try:
            new_total = (df_indexed.loc[purchase_ids, "scores"] @
                         self.purchases.loc[purchase_ids, :]) \
                        + self.direct_impacts.sum().sum()
        except KeyError as e:
            print(f"  KeyError during update check: {e}")
            return False

        new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
        print(f"  New footprint estimate: {new_fp:.6f} (Previous: {self.latest_update})")

        if self.latest_update is None or not np.isclose(self.latest_update, new_fp, atol=1e-6):
            print(f"  -> Update required for {self.name}")
            # call update_footprint on the original DB
            self.update_footprint(footprint_db_2024, footprint_db_2023, last_year_sales_db)
            return True
        else:
            print(f"  -> No update needed for {self.name}")
            return False


    def get_average_footprint(self, class_code, last_year_sales_db, footprint_db_2023):
        """
        Calculate the weighted average footprint per unit for all primary products
        with the given classification code from last year's sales, adjusting for function_output.
        """
        # Get last-year sales data for products with this classification code
        sales_info = last_year_sales_db.get_sales_data_by_class_code(class_code)

        if sales_info.empty:
            print(f"  No sales data found for class code {class_code}")
            return 0  # No data available

        # Ensure the footprint database IDs are integers for proper matching
        footprint_db_2023.data["id"] = footprint_db_2023.data["id"].astype(int)

        # Merge sales data with footprint data to match products and footprints
        merged_data = sales_info.merge(footprint_db_2023.data, on="id", how="left").fillna(0)

        # Get function_output from last_year_sales_db
        function_outputs = {}
        for product_id in merged_data['id'].unique():
            product_row = last_year_sales_db.data[last_year_sales_db.data['id'] == product_id]
            if not product_row.empty:
                function_outputs[product_id] = product_row['function_output'].iloc[0]

        # Calculate weighted impact, incorporate conversion factor
        total_weighted_impact = 0
        total_sales_volume = 0

        for index, row in merged_data.iterrows():
            product_id = row['id']
            sales_volume = row['sales_volume']
            footprint_score = row['scores']
            conversion_factor = function_outputs.get(product_id, 1)  # get the conversion factor

            adjusted_impact = sales_volume * footprint_score * conversion_factor
            total_weighted_impact += adjusted_impact
            total_sales_volume += sales_volume * conversion_factor

        # Avoid division by zero
        weighted_average = total_weighted_impact / total_sales_volume if total_sales_volume > 0 else 0

        print(f"  Weighted avg. footprint for {class_code}: {weighted_average:.4f} (Total sales: {total_sales_volume})")
        return weighted_average

    def report_footprint(self, footprint_db_2024):
        """Report the latest footprint for each product to the footprint database."""
        print("report_footprint function called.")
        for product in self.products:
            # Extract the float value from the Pandas Series
            footprint_value = product.footprint if isinstance(product.footprint, float) else product.footprint.values[0]
            footprint_db_2024.report(product.product_id, footprint_value)

In [102]:

class SalesDatabase:
    """
    Stores sales volume data for different product IDs per year.
    """

    def __init__(self, year, classification_db):
        self.year = year  # The year the sales data is from
        self.classification_db = classification_db  # Reference to classification database
        self.data = pd.DataFrame(columns=["id", "class_code", "sales_volume", "unit"])

    def add_sales(self, product: Product, sales_volume):
        """
        Add sales data for a given product, ensuring the unit matches classification database.
        """
        # Fetch classification info dynamically
        class_info = self.classification_db.data.loc[self.classification_db.data["class_code"] == product.class_code]

        if class_info.empty:
            raise ValueError(f"Classification for {product.class_code} not found.")

        correct_unit = class_info.iloc[0]["unit"]
        if product.unit != correct_unit:
            raise ValueError(f"Unit mismatch: {product.unit} (product) vs {correct_unit} (expected).")

        # Add sales entry
        new_entry = pd.DataFrame({
            "id": [product.product_id],
            "class_code": [product.class_code],
            "sales_volume": [sales_volume],  # the sales in the unit corresponding to the reference unit of the classification code
            "unit": [correct_unit],  # unit from classification code
            "function_output": [product.function_output],  # the amount of function, as defined by the classification code, per monetary unit. Example: Each 1$ of crushed granite purchased gives 0.025 m3 granite, m3 being the unit for crushed granite.
        })

        self.data = pd.concat([self.data, new_entry], ignore_index=True)

    def get_sales_data_by_class_code(self, class_code):
        """
        Retrieve sales data for a given class code.

        Args:
            class_code (str): The class code.

        Returns:
            pd.DataFrame: The sales data for the given class code.
        """
        print(f"  get_sales_data_by_class_code called for class code: '{class_code}'")  # debugging
        print(f"  all class codes in sales data: {self.data['class_code'].unique()}")  # debugging
        print(f"  Data types of class_code column: {self.data['class_code'].dtype}")  # debugging
        print(f"  Data type of class_code argument: {type(class_code)}")  # debugging
        sales_data = self.data[self.data["class_code"] == class_code]
        print(f"  Sales Data retrieved:\n{sales_data}")  # debugging
        return sales_data

    def get_sales_data(self, product_id):
        """
        Retrieve sales data for a given product ID.

        Args:
            product_id (int): The product ID.

        Returns:
            pd.DataFrame: The sales data for the given product.
        """
        return self.data.loc[self.data["id"] == product_id]

In [103]:
class System:
    """
    Represents the entire system of companies, products, and interactions.
    """

    def __init__(self, num_companies, num_products, classification_db):
        self.num_companies = num_companies
        self.num_products = num_products
        self.classification_db = classification_db
        self.companies = {}

        # Initialize key data structures
        self.use_data = np.ones((num_products, num_companies))
        np.fill_diagonal(self.use_data, 0)  # Avoid self-purchase

        self.margins_data = [random.uniform(1.28, 1.89) for _ in range(num_companies)]
        self.supply_data = np.zeros((num_products, num_companies))
        self.emissions_data = np.zeros((1, num_companies))

        self.create_companies()
        self.assign_products_to_companies()

    def create_companies(self):
        """Initializes companies with purchases, sales, and direct emissions."""
        for i in range(self.num_companies):
            purchases = pd.Series(self.use_data[:, i], index=[j + 1 for j in range(self.num_products)])  # Use integer indices
            sales = np.zeros(self.num_products)  # Start with zero sales
            direct_impacts = pd.DataFrame({"kg CO2eq": [np.random.randint(0, 659)]})

            company = Company(f'Company {i+1}', 2023, purchases, sales, direct_impacts)
            self.companies[f'Company_{i+1}'] = company

    def assign_products_to_companies(self):
        """Assigns products to each company and calculates sales correctly."""
        co_product_mapping = {
            0: ("Granite tiles", "Crushed granite", "HS 2517"),
            1: ("Biodiesel", "Glycerol", "HS 2905"),
            2: ("Hot-Rolled Steel", "Blast Furnace Slag", "HS 2618"),
        }

        for i in range(self.num_companies):
            company = self.companies[f'Company_{i+1}']
            margin = self.margins_data[i]

            # Initialize sales DataFrame with the correct index (products) and one column
            sales_dict = {}

            # **Companies 1-10 get one primary product**
            if i < 10:
                product_id = i
                total_sales = self.use_data[:, i].sum() * margin

                product = Product(
                    product_id + 1,
                    f"Product {product_id + 1}",
                    "unit",
                    company,
                    "class_code",
                    "primary",
                    total_sales,
                    self.classification_db
                )
                company.add_product(product)

                # ✅ Store sales in dictionary for correct alignment
                sales_dict[product_id + 1] = total_sales

            # **Companies 11-13 get a main product + co-product**
            elif (i - 10) in co_product_mapping:
                main_name, co_name, co_class = co_product_mapping[i - 10]
                base_product_id = 10 + (i - 10) * 2
                co_product_id = base_product_id + 1

                total_sales = self.use_data[:, i].sum() * margin
                co_product_sales = total_sales * 0.5  # 50% of total sales
                main_product_sales = total_sales  # The full total sales

                # **Main product**
                main_product = Product(
                    base_product_id + 1,
                    main_name,
                    "unit",
                    company,
                    "class_code",
                    "primary",
                    main_product_sales,
                    self.classification_db
                )
                company.add_product(main_product)

                # **Co-product**
                co_product = Product(
                    co_product_id + 1,
                    co_name,
                    "unit",
                    company,
                    co_class,  # The assigned classification code
                    "secondary",
                    co_product_sales,
                    self.classification_db
                )
                company.add_product(co_product)

                # ✅ Store sales for both in dictionary
                sales_dict[base_product_id + 1] = main_product_sales
                sales_dict[co_product_id + 1] = co_product_sales

            # ✅ Convert dictionary to properly formatted DataFrame (only with necessary columns)
            company.sales = pd.DataFrame.from_dict(sales_dict, orient="index", columns=["Sales"])

    def print_company_info(self):
        """Prints sales and purchases for debugging."""
        for name, company in self.companies.items():
            print(f"\n--- {name} ---")
            print("Purchases:\n", company.purchases)
            print("Sales:\n", company.sales)
            print("Direct emissions:\n", company.direct_impacts)




In [104]:
# Initialize the classification database with dummy data
classification_data = [
    {
        "class_code": "HS 2517",
        "class_name": "Pebbles, gravel, broken or crushed stone, etc.",
        "function": "Fill a space and provide mechanical stability",
        "unit": "m3"
    },
    {
        "class_code": "HS 2905",
        "class_name": "Acyclic alcohols and their derivatives",
        "function": "Acts as a chemical feedstock for synthesis and retains moisture as a humectant",
        "unit": "kg"
    },
    {
        "class_code": "HS 2618",
        "class_name": "Granulated slag (slag sand) from the manufacture of iron or steel",
        "function": "Acts as a binder and improves mechanical properties in road bases",
        "unit": "kg"
    }
]

# Create an instance of the classification database
classification_db = ClassificationDatabase(classification_data)

In [105]:
# Initialize last year's sales database
last_year_sales_db = SalesDatabase(year=2023, classification_db=classification_db)

# Create a sample product
product12 = Product(12, "Crushed Granite", "m3", "Company 11", "HS 2517", "primary", 0.5, classification_db)
product14 = Product(14, "Glycerol", "kg", "Company 12", "HS 2905", "primary", 0.05, classification_db)
product16 = Product(16, "Blast Furnace Slag", "kg", "Company 13", "HS 2618", "primary", 0.7, classification_db)

# Add sales for this product
last_year_sales_db.add_sales(product12, 1000)
last_year_sales_db.add_sales(product14, 2000)
last_year_sales_db.add_sales(product16, 3000)

# Retrieve sales data for product 1
print(last_year_sales_db.get_sales_data_by_class_code("HS 2517"))
print(last_year_sales_db.get_sales_data_by_class_code("HS 2905"))
print(last_year_sales_db.get_sales_data_by_class_code("HS 2618"))

my_system = System(num_companies=13, num_products=16, classification_db=classification_db)
my_system.print_company_info()  # Debug output

# Initialize footprint databases
footprint_db_2023 = FootprintDatabase(year=2023)
footprint_db_2024 = FootprintDatabase(year=2024)

print(f"footprint_db_2024 created at: {id(footprint_db_2024)}")  # Debugging

# Initialize footprint database with dummy values (2023)
for company in my_system.companies.values():
    for product in company.products:
        footprint_db_2023.report(product.product_id, random.uniform(0, 100))  # Random values between 0 and 100

# Initialize footprint database with zero values (2024)
for company in my_system.companies.values():
    for product in company.products:
        print(f"footprint_db_2024 before report: {id(footprint_db_2024)}")  # Debugging
        footprint_db_2024.report(product.product_id, 0)

# example.
# random_company.update_footprint(footprint_db_2023, last_year_sales_db)

  get_sales_data_by_class_code called for class code: 'HS 2517'
  all class codes in sales data: ['HS 2517' 'HS 2905' 'HS 2618']
  Data types of class_code column: object
  Data type of class_code argument: <class 'str'>
  Sales Data retrieved:
   id class_code sales_volume unit  function_output
0  12    HS 2517         1000   m3              0.5
   id class_code sales_volume unit  function_output
0  12    HS 2517         1000   m3              0.5
  get_sales_data_by_class_code called for class code: 'HS 2905'
  all class codes in sales data: ['HS 2517' 'HS 2905' 'HS 2618']
  Data types of class_code column: object
  Data type of class_code argument: <class 'str'>
  Sales Data retrieved:
   id class_code sales_volume unit  function_output
1  14    HS 2905         2000   kg             0.05
   id class_code sales_volume unit  function_output
1  14    HS 2905         2000   kg             0.05
  get_sales_data_by_class_code called for class code: 'HS 2618'
  all class codes in sales dat

In [106]:
import random
import time

# Create a list of company names
company_names = list(my_system.companies.keys())

# Initialize a flag to keep track of updates
updates_completed = False

start_time = time.time()
iteration = 0
# Initialize footprint database with initial values (optional)
for company in my_system.companies.values():
    for product in company.products:
        footprint_db_2024.report(product.product_id, 0) # Initialize with 0 or estimated starting values

# Outer loop ensures each company is checked at least once
while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    
    #Create list to keep track of companies to check
    companies_to_check = list(company_names)
    companies_not_checked = list(company_names)

    # Reset the flag for the current iteration
    any_company_updated = False

    # Randomly select companies to check
    while companies_not_checked:
        random_company_name = random.choice(companies_to_check)
        random_company = my_system.companies[random_company_name]

        print(f"Checking company: {random_company_name}")

        # Use check_update_needed to determine if an update is needed
        if random_company.check_update_needed(footprint_db_2024, last_year_sales_db):
            print(f"Company {random_company_name} was updated.")
            any_company_updated = True

        # Remove the checked company from the list
        if random_company_name in companies_not_checked:
            companies_not_checked.remove(random_company_name)
            
    updates_completed = not any_company_updated

# Simulation finished
print("\nUpdates completed.")

# End the timer
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Time taken: {elapsed_time:.2f} seconds")

# Print final footprints for each company
print("\nFinal Footprint Reports:")
for company_name, company in my_system.companies.items():
    print(f"{company_name}: {company.latest_update:.4f}")
total_updates = sum(company.num_updates for company in my_companies.values())
average_updates_per_company = total_updates / num_companies
#for company_name, company in my_companies.items():
#    print(f"{company_name} updates: {company.num_updates}")
print(f"Total updates: {total_updates}")
    
# After the main while loop, force some additional updates
for i in range(3):  # Try 3 extra forced updates
    print(f"\n--- Forced Update Iteration {i+1} ---")
    for company_name, company in my_system.companies.items():
        company.update_footprint(footprint_db_2024, footprint_db_2023, last_year_sales_db)


--- Iteration 1 ---
Checking company: Company_11

Checking updates for Company 11
  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns available: ['index', 'id', 'impact_ids', 'scores']
  New footprint estimate: 8.556209 (Previous: None)
  -> Update required for Company 11
Calculating footprint for Company 11
  Expected product IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Footprint DB IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Initial total impacts: 0    263.0
Name: scores, dtype: float64

  Processing product: Granite tiles (primary)

  Processing product: Crushed granite (secondary)
  get_sales_data_by_class_code called for class code: 'HS 2517'
  all class codes in sales data: ['HS 2517' 'HS 2905' 'HS 2618']
  Data types of class_code column: object
  Data type of class_code argument: <class 'str'>
  Sales Data retrieved:
   id class_code sales_volume unit  function_output
0  12    HS 2517         

C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.latest_update = float(num / den)
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprec

C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.latest_update = float(num / den)
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprec

  New footprint estimate: 49.978337 (Previous: 46.908618435198505)
  -> Update required for Company 13
Calculating footprint for Company 13
  Expected product IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Footprint DB IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Initial total impacts: 0    1463.162184
Name: scores, dtype: float64

  Processing product: Hot-Rolled Steel (primary)

  Processing product: Blast Furnace Slag (secondary)
  get_sales_data_by_class_code called for class code: 'HS 2618'
  all class codes in sales data: ['HS 2517' 'HS 2905' 'HS 2618']
  Data types of class_code column: object
  Data type of class_code argument: <class 'str'>
  Sales Data retrieved:
   id class_code sales_volume unit  function_output
2  16    HS 2618         3000   kg              0.7
  Weighted avg. footprint for HS 2618: 1.7237 (Total sales: 2100.0)
    Assigned footprint (secondary): 16.820942697650764
    Recalculated footprint (primary): 0    66.5

C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.latest_update = float(num / den)
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:83: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.latest_update = float(num / den)
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future.

...

  Initial total impacts: 0    1849.581241
Name: scores, dtype: float64

  Processing product: Product 5 (primary)
    Recalculated footprint (primary): 0    83.19931
Name: scores, dtype: float64
report_footprint function called.
  Final footprint for Company 5: 83.19931002865205
Company Company_5 was updated.
Checking company: Company_13

Checking updates for Company 13
  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns available: ['index', 'id', 'impact_ids', 'scores']
  New footprint estimate: 64.766718 (Previous: 64.7657845164275)
  -> Update required for Company 13
Calculating footprint for Company 13
  Expected product IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Footprint DB IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Initial total impacts: 0    1896.105771
Name: scores, dtype: float64

  Processing product: Hot-Rolled Steel (primary)

  Processing product: Blast Furnace Slag (secondary)
  get_s

C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Ca

  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns available: ['index', 'id', 'impact_ids', 'scores']
  New footprint estimate: 55.079588 (Previous: 55.079588375765795)
  -> No update needed for Company 11
Checking company: Company_4

Checking updates for Company 4
  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns available: ['index', 'id', 'impact_ids', 'scores']
  New footprint estimate: 68.961849 (Previous: 68.96180835029311)
  -> No update needed for Company 4
Checking company: Company_6

Checking updates for Company 6
  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns available: ['index', 'id', 'impact_ids', 'scores']
  New footprint estimate: 58.785246 (Previous: 58.78504949737915)
  -> No update needed for Company 6
Checking company: Company_9

Checking updates for Company 9
  self.purchases.index: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Columns

C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  new_fp = float(new_total / self.sales.sum().sum()) if self.sales.sum().sum() else 0
C:\Users\marit.rognan\AppData\Local\Temp\ipykernel_4896\509280182.py:112: FutureWarning: Ca

  Expected product IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Footprint DB IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Initial total impacts: 0    1873.807518
Name: scores, dtype: float64

  Processing product: Product 9 (primary)
    Recalculated footprint (primary): 0    67.989918
Name: scores, dtype: float64
report_footprint function called.
  Final footprint for Company 9: 67.98991809391411
Calculating footprint for Company 10
  Expected product IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Footprint DB IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  Initial total impacts: 0    1871.118536
Name: scores, dtype: float64

  Processing product: Product 10 (primary)
    Recalculated footprint (primary): 0    85.679205
Name: scores, dtype: float64
report_footprint function called.
  Final footprint for Company 10: 85.67920474529024
Calculating footprint for Company 11
  Expected product IDs: [1, 2,

In [107]:
# Print final footprints for each company
print("\nFinal Footprint Reports:")
for company_name, company in my_system.companies.items():
    print(f"{company_name}: {company.latest_update:.4f}")


Final Footprint Reports:
Company_1: 94.9172
Company_2: 57.8355
Company_3: 76.8780
Company_4: 68.9622
Company_5: 83.2002
Company_6: 58.7856
Company_7: 61.2780
Company_8: 89.7721
Company_9: 67.9901
Company_10: 85.6794
Company_11: 55.0799
Company_12: 44.3804
Company_13: 64.7674


In [108]:
print("\nFootprint Database 2024:")
print(footprint_db_2024.data)

print("\nFootprint Database 2023:")
print(footprint_db_2023.data)
# Assuming you have last_year_sales_db, footprint_db_2023, and classification_db defined

# Print last_year_sales_db
print("Last Year Sales Data:")
print(last_year_sales_db.data)  # Assuming last_year_sales_db has a suitable __str__ or representation

# Print classification_db
print("\nClassification Data:")
print(classification_db.data)  # Print the DataFrame within classification_db



Footprint Database 2024:
    id impact_ids      scores
0    1        CO2   94.917152
1    2        CO2   57.835532
2    3        CO2   76.878040
3    4        CO2   68.962189
4    5        CO2   83.200247
5    6        CO2   58.785609
6    7        CO2   61.277959
7    8        CO2   89.772085
8    9        CO2   67.990097
9   10        CO2   85.679418
10  11        CO2  -26.238398
11  12        CO2  217.716492
12  13        CO2 -228.323472
13  14        CO2  589.788210
14  15        CO2   88.740651
15  16        CO2   16.820943

Footprint Database 2023:
    id impact_ids     scores
0    1        CO2  32.532449
1    2        CO2  71.170144
2    3        CO2  24.262642
3    4        CO2  87.082376
4    5        CO2  94.057087
5    6        CO2  87.932043
6    7        CO2  12.724521
7    8        CO2  49.809650
8    9        CO2  15.769130
9   10        CO2  93.568565
10  11        CO2  96.724195
11  12        CO2  21.248987
12  13        CO2  81.064319
13  14        CO2  54.986283
14 

In [111]:
import numpy as np

for cname, company in my_system.companies.items():
    print(f"\n--- Proof‑check for {cname} ---")
    
    # 1) Embodied impacts of purchases
    scores = footprint_db_2024.data.set_index("id")["scores"].astype(float)
    purch = company.purchases.iloc[:, 0]  # assumes single‐column purchases
    in_emb = float(scores.loc[purch.index].dot(purch))
    
    # 2) Direct emissions
    in_direct = float(company.direct_impacts.sum().sum())
    
    # 3) Total carbon in
    carbon_in = in_emb + in_direct
    
    # 4) Carbon out (embodied in sales only)
    total_sales = float(company.sales["Sales"].sum())
    out_emb = float(company.latest_update * total_sales)
    
    # 5) Balance check
    balanced = np.isclose(carbon_in, out_emb, atol=1e-6)
    
    # Print everything
    print(f"Purchases embodied (in_emb):     {in_emb:.6f}")
    print(f"Direct emissions   (in_direct):  {in_direct:.6f}")
    print(f"Total carbon in    (carbon_in):  {carbon_in:.6f}")
    print(f"Sales sum          (total_sales):{total_sales:.6f}")
    print(f"Carbon out embodied (out_emb):   {out_emb:.6f}")
    print(f"Balance?                         {balanced}")



--- Proof‑check for Company_1 ---
Purchases embodied (in_emb):     1308.885601
Direct emissions   (in_direct):  586.000000
Total carbon in    (carbon_in):  1894.885601
Sales sum          (total_sales):19.963566
Carbon out embodied (out_emb):   1894.884848
Balance?                         True

--- Proof‑check for Company_2 ---
Purchases embodied (in_emb):     1345.967221
Direct emissions   (in_direct):  38.000000
Total carbon in    (carbon_in):  1383.967221
Sales sum          (total_sales):23.929348
Carbon out embodied (out_emb):   1383.966558
Balance?                         True

--- Proof‑check for Company_3 ---
Purchases embodied (in_emb):     1326.924713
Direct emissions   (in_direct):  416.000000
Total carbon in    (carbon_in):  1742.924713
Sales sum          (total_sales):22.671287
Carbon out embodied (out_emb):   1742.924138
Balance?                         True

--- Proof‑check for Company_4 ---
Purchases embodied (in_emb):     1334.840564
Direct emissions   (in_direct):  535

In [112]:
import numpy as np

for cname, company in my_system.companies.items():
    # Identify secondaries and primaries
    secondaries = [p for p in company.products if p.primary_secondary=="secondary"]
    primaries   = [p for p in company.products if p.primary_secondary=="primary"]
    if not secondaries:
        continue  # skip companies without co-products

    print(f"\n=== Proof‑check for {cname} (with co‑products) ===")

    # 1) Carbon in
    scores    = footprint_db_2024.data.set_index("id")["scores"].astype(float)
    purch     = company.purchases.iloc[:,0]
    in_emb    = float(scores.loc[purch.index].dot(purch))
    in_direct = float(company.direct_impacts.sum().sum())
    carbon_in = in_emb + in_direct
    print(f" carbon_in = embodied({in_emb:.3f}) + direct({in_direct:.3f}) = {carbon_in:.3f}")

    # 2) Secondaries: assigned vs expected
    total_sec_impact = 0.0
    print("\n-- Secondary products --")
    for p in secondaries:
        # assigned footprint/unit
        fp = p.footprint
        if hasattr(fp, "iloc"):
            fp = float(fp.iloc[0])
        sv = float(company.sales.loc[p.product_id, "Sales"])
        # expected per‑unit
        avg_sub  = company.get_average_footprint(p.class_code, last_year_sales_db, footprint_db_2023)
        expected = avg_sub * p.function_output
        impact   = fp * sv
        total_sec_impact += impact

        print(f" {p.name}:")
        print(f"   assigned fp/unit = {fp:.6f}")
        print(f"   expected fp/unit = {expected:.6f}")
        print(f"   match?           = {np.isclose(fp, expected, atol=1e-6)}")
        print(f"   sales_vol={sv:.3f} → impact={impact:.3f}")

    # 3) Primaries: compute expected footprint/unit after subtracting secondaries
    total_primary_sales = sum(float(company.sales.loc[p.product_id, "Sales"]) for p in primaries)
    expected_primary_fp = (carbon_in - total_sec_impact) / total_primary_sales
    print(f"\n-- Primary products --")
    print(f" total_secondary_impact = {total_sec_impact:.3f}")
    print(f" expected_primary_fp    = (carbon_in - sec_imp)/sum(sales_prim) = {expected_primary_fp:.6f}")

    for p in primaries:
        fp = p.footprint
        if hasattr(fp, "iloc"):
            fp = float(fp.iloc[0])
        print(f" {p.name}: assigned fp/unit = {fp:.6f} → match? {np.isclose(fp, expected_primary_fp, atol=1e-6)}")

    # 4) Final carbon balance
    combined_out = total_sec_impact + expected_primary_fp * total_primary_sales
    print(f"\n-- Carbon balance --")
    print(f"  combined_out = sec_imp({total_sec_impact:.3f}) + prim_imp({expected_primary_fp*total_primary_sales:.3f}) = {combined_out:.3f}")
    print(f"  carbon_in    = {carbon_in:.3f}")
    print(f"  balance?     = {np.isclose(combined_out, carbon_in, atol=1e-6)}")



=== Proof‑check for Company_11 (with co‑products) ===
 carbon_in = embodied(1430.041) + direct(263.000) = 1693.041

-- Secondary products --
  get_sales_data_by_class_code called for class code: 'HS 2517'
  all class codes in sales data: ['HS 2517' 'HS 2905' 'HS 2618']
  Data types of class_code column: object
  Data type of class_code argument: <class 'str'>
  Sales Data retrieved:
   id class_code sales_volume unit  function_output
0  12    HS 2517         1000   m3              0.5
  Weighted avg. footprint for HS 2517: 21.2490 (Total sales: 500.0)
 Crushed granite:
   assigned fp/unit = 217.716492
   expected fp/unit = 217.716492
   match?           = True
   sales_vol=10.246 → impact=2230.717

-- Primary products --
 total_secondary_impact = 2230.717
 expected_primary_fp    = (carbon_in - sec_imp)/sum(sales_prim) = -26.238394
 Granite tiles: assigned fp/unit = -26.238398 → match? True

-- Carbon balance --
  combined_out = sec_imp(2230.717) + prim_imp(-537.676) = 1693.041
  carbo

class Company:
    """
    Represents a company with multi-level structure for environmental footprint reporting.
    """

    def __init__(self, name, year, purchases=None, sales=None, direct_impacts=None):
        self.name = name  # Name of the company
        self.year = year  # Reporting year
        self.purchases = pd.DataFrame(purchases) if purchases is not None else pd.DataFrame()
        self.sales = pd.DataFrame(sales) if sales is not None else pd.DataFrame()
        self.direct_impacts = pd.DataFrame(direct_impacts) if direct_impacts is not None else pd.DataFrame()
        self.sub_companies = []  # List of child Company objects
        self.products = []  # List of Product objects associated with this company
        self.latest_update = None
        self.num_updates = 0

    def add_product(self, product):
        """Add a product associated with this company."""
        self.products.append(product)

    def update_footprint(self, footprint_db_2024, footprint_db_2023, last_year_sales_db):
        """
        Update the company's footprint based on carbon balance.
        """
        print(f"Calculating footprint for {self.name}")

        # Convert purchase index to integer list
        product_ids = list(self.purchases.index.astype(int))

        # Ensure integer IDs in footprint_db_2024
        if 'id' not in footprint_db_2024.data.columns:
            footprint_db_2024.data.reset_index(inplace=True, drop=True)  # reset index if 'id' is not a column
        else:
            footprint_db_2024.data.reset_index(inplace=True, drop=True)

        if 'id' in footprint_db_2024.data:
            footprint_db_2024.data["id"] = footprint_db_2024.data["id"].astype(int)

        print(f"  Expected product IDs: {product_ids}")
        print(f"  Footprint DB IDs: {footprint_db_2024.data['id'].tolist()}")

        # Ensure IDs exist in footprint_db before selecting
        missing_ids = [pid for pid in product_ids if pid not in footprint_db_2024.data["id"].values]

        total_impacts = (
            footprint_db_2024.data.set_index("id").loc[product_ids, "scores"] @ self.purchases
        ) + self.direct_impacts.sum().sum()

        print(f"  Initial total impacts (before system expansion): {total_impacts}")

        # Temporary storage for primary product impacts
        primary_product_impacts = {}

        # Calculate and assign footprints **individually per product**
        for product in self.products:
            print(f"\n  Processing product: {product.name} ({product.primary_secondary})")

            # Calculate product's sales volume
            product_sales_volume = self.sales.loc[product.product_id, "Sales"] if product.product_id in self.sales.index else 0

            if product.primary_secondary == "primary":
                # Initialize primary product impact with total company impact
                primary_product_impacts[product.product_id] = total_impacts
                product_footprint = total_impacts / product_sales_volume if product_sales_volume > 0 else 0
                product.footprint = product_footprint
                print(f"    Assigned footprint (primary): {product.footprint}")

            elif product.primary_secondary == "secondary":
                # Get weighted average footprint of substituted product
                avg_substituted_impact = self.get_average_footprint(product.class_code, last_year_sales_db, footprint_db_2023)

                # Calculate substituted impact
                substituted_impact = avg_substituted_impact * product_sales_volume * product.function_output

                # Subtract substituted impact from the corresponding primary product's impacts
                for primary_product in self.products:
                    if primary_product.primary_secondary == "primary" and primary_product.product_id != product.product_id:  # Avoid subtracting from itself
                        primary_product_impacts[primary_product.product_id] -= substituted_impact

                product.footprint = substituted_impact / product_sales_volume if product_sales_volume > 0 else 0
                print(f"    Assigned footprint (secondary): {product.footprint}")

        # Recalculate primary product footprints after substitution
        for product in self.products:
            if product.primary_secondary == "primary":
                product.footprint = primary_product_impacts[product.product_id] / product_sales_volume if product_sales_volume > 0 else 0
                print(f"    Recalculated footprint (primary): {product.footprint}")

        # Store company-level footprint as the **weighted average** of all product footprints
        self.latest_update = float(sum(float(p.footprint) * p.function_output for p in self.products) / sum(p.function_output for p in self.products))
        self.report_footprint(footprint_db_2024)

        print(f"  Final footprint for {self.name}: {self.latest_update}")

    def check_update_needed(self, footprint_db_2024, last_year_sales_db):
        """
        Check if a footprint update is needed for any products in the company.
        """
        print(f"\nChecking updates for {self.name}")

        print(f"  self.purchases.index: {self.purchases.index}")  # Print the index

        # Explicitly convert to integer index
        purchase_ids = self.purchases.index.astype(int)

        # Explicitly convert footprint_db_2024.data['id'] to integers
        if 'id' in footprint_db_2024.data:
            footprint_db_2024.data['id'] = footprint_db_2024.data['id'].astype(int)

        # Add print statement to check columns
        print(f"  Columns before ID check: {footprint_db_2024.data.columns}")

        footprint_ids = footprint_db_2024.data["id"].astype(int)

        # Explicitly set the index of footprint_db_2024.data
        footprint_db_2024.data.set_index('id', inplace=True)  # use inplace=True

        # Compute new footprint
        try:
            new_total_impacts = (footprint_db_2024.data.loc[purchase_ids, "scores"] @ self.purchases) + self.direct_impacts.sum().sum()
        except KeyError as e:
            print(f"  KeyError: {e}")
            return False

        # Track if any product needs an update
        any_product_updated = False

        # Process all products and check for individual updates
        for product in self.products:
            print(f"  Checking product: {product.name}")

            # Calculate product's sales volume
            product_sales_volume = self.sales.loc[product.product_id, "Sales"] if product.product_id in self.sales.index else 0

            if product.primary_secondary == "primary":
                new_product_footprint = new_total_impacts / product_sales_volume if product_sales_volume > 0 else 0
            else:
                avg_substituted_impact = self.get_average_footprint(product.class_code, last_year_sales_db, footprint_db_2023)
                new_product_footprint = (new_total_impacts - avg_substituted_impact * product_sales_volume) / product_sales_volume if product_sales_volume > 0 else 0

            # Check for individual product update
            previous_footprint = getattr(product, 'previous_footprint', None)  # Get previous footprint, default to None
            if previous_footprint is None or not np.isclose(previous_footprint, new_product_footprint, atol=1e-6):
                print(f"    -> Update required for product: {product.name}")
                any_product_updated = True
                product.previous_footprint = new_product_footprint  # Store the new footprint
            else:
                print(f"    -> No update needed for product: {product.name}")

        # Trigger company update if any product needs it
        if any_product_updated:
            print(f"  -> Update required for {self.name}")
            self.update_footprint(footprint_db_2024, footprint_db_2023, last_year_sales_db)
            return True
        else:
            print(f"  -> No update needed for {self.name}")
            return False

    def get_average_footprint(self, class_code, last_year_sales_db, footprint_db_2023):
        """
        Calculate the weighted average footprint per unit for all primary products
        with the given classification code from last year's sales, adjusting for function_output.
        """
        # Get last-year sales data for products with this classification code
        sales_info = last_year_sales_db.get_sales_data_by_class_code(class_code)

        if sales_info.empty:
            print(f"  No sales data found for class code {class_code}")
            return 0  # No data available

        # Ensure the footprint database IDs are integers for proper matching
        footprint_db_2023.data["id"] = footprint_db_2023.data["id"].astype(int)

        # Merge sales data with footprint data to match products and footprints
        merged_data = sales_info.merge(footprint_db_2023.data, on="id", how="left").fillna(0)

        # Get function_output from last_year_sales_db
        function_outputs = {}
        for product_id in merged_data['id'].unique():
            product_row = last_year_sales_db.data[last_year_sales_db.data['id'] == product_id]
            if not product_row.empty:
                function_outputs[product_id] = product_row['function_output'].iloc[0]

        # Calculate weighted impact, incorporate conversion factor
        total_weighted_impact = 0
        total_sales_volume = 0

        for index, row in merged_data.iterrows():
            product_id = row['id']
            sales_volume = row['sales_volume']
            footprint_score = row['scores']
            conversion_factor = function_outputs.get(product_id, 1)  # get the conversion factor

            adjusted_impact = sales_volume * footprint_score * conversion_factor
            total_weighted_impact += adjusted_impact
            total_sales_volume += sales_volume * conversion_factor

        # Avoid division by zero
        weighted_average = total_weighted_impact / total_sales_volume if total_sales_volume > 0 else 0

        print(f"  Weighted avg. footprint for {class_code}: {weighted_average:.4f} (Total sales: {total_sales_volume})")

        return weighted_average

    def report_footprint(self, footprint_db_2024):
        """Report the latest footprint for each product to the footprint database."""
        print("report_footprint function called.")
        for product in self.products:
            # Extract the float value from the Pandas Series
            footprint_value = product.footprint if isinstance(product.footprint, float) else product.footprint.values[0]
            footprint_db_2024.report(product.product_id, footprint_value)
